# Python Insights - Analisando Dados com Python

### Case - Cancelamento de Clientes

Simulação de uma empresa com mais de 50 mil clientes que percebeu que a maioria de seus clientes são inativos, ou seja,  já cancelaram o serviço.

Precisando melhorar seus resultados, ela quer conseguir entender os principais motivos desses cancelamentos e quais as ações mais eficientes para reduzir esse número.



# Importar bibliotecas necessárias e visualizar a base de dados

In [ ]:

import pandas as pd
import plotly.express as px
tabela=pd.read_csv("cancelamentos.csv")
display(tabela)



# Visualização e tratamento dos dados

In [ ]:

display(tabela.info())


tabela=tabela.drop(columns="CustomerID") # Remover coluna desnecessária
tabela=tabela.dropna() # Remover linhas com valores nulos
tabelas=tabela.drop_duplicates() # Remover linhas duplicadas
tabela=tabela.reset_index(drop=True) # Resetar índices após remoção de linhas
display(tabela.info())

# Análise inicial dos dados

In [ ]:
colunas_analise = tabela.columns 

for coluna in colunas_analise:
        print(f"Analisando: {coluna}")
        
        
        tabela_resumo = pd.DataFrame({
            "Total": tabela[coluna].value_counts(),
            "Porcentagem": tabela[coluna].value_counts(normalize=True).map("{:.2%}".format)
        })
        
        #se for numérica, ordena pelo valor
        if pd.api.types.is_numeric_dtype(tabela[coluna]):
            display(tabela_resumo.sort_index())
        # Se for categoria, ordena pelo padrão
        else:
            display(tabela_resumo)

# Pontos importantes:

56% dos clientes são Masculinos
Menos de 20% dos nosso clientes tem menos de um ano conosco
Mais de 50% dos nossos clientes ligam de 0 a 3 vezes para o Callcenter
Menos de 10% dos clientes com menos de 2 dias de atraso.
Distribuição entre as assinaturas é equilibradas (em torno de 33% para cada)
40% dos contratos são anuais e 20% são mensais, aproximadamente.
56% dos clientes cancelaram o plano

# Análise exploratória dos dados com gráficos

In [ ]:


for coluna in tabela.columns:
    if coluna != "cancelou":
        print(f"Analisando a coluna: {coluna}")

        grafico=px.histogram(tabela, x=coluna, color="cancelou", barmode="group", title=f"Distribuição de {coluna} por Cancelamento")
        grafico.show()
        print("-" * 50)  # Linha separadora visual



# Insights

Todos os clientes com mais de 50 anos cancelaram.
Clientes femininos tem alta taxa de cancelamento.
Clientes com tempo de contrato entre 0-6 e 12-24 meses tem alta taxa de cancelamento em relação aos demais
Alta taxa de cancelamento em clientes com baixa frequência de uso
Clientes que ligam mais de 5 vezes para o callcenter cancelam.
Clientes com mais de 20 dias de atraso cancelam.
Todos os clientes com contrato mensal, cancelam.
Clientes com mais de 15 meses sem interação, tem alta taxa de cancelamento.

# Sugestões de "ataque" aos ofensores




Todos os clientes com mais de 50 anos cancelaram.
-> Rever nossa plataforma e reavaliar o UX para idade

Clientes femininos tem alta taxa de cancelamento.
-> Pesquisa de satisfação com o público feminino para reavaliar.

Clientes com tempo de contrato entre 0-6 e 12-24 meses tem alta taxa de cancelamento em relação aos demais
-> Clientes com menos de um ano representam menos de 20% de nossa base, o que não se mostra um ofensor tão grande. Porém, é importante que acompanhemos esse parâmetro para reavaliar os produtos oferecidos que fidelizam o cliente por mais tempo.

Alta taxa de cancelamento em clientes com baixa frequência de uso
-> Creio que isso seja normal...

Clientes que ligam mais de 5 vezes para o callcenter cancelam.
-> Considerando que Mais de 50% dos nossos clientes ligam de 0 a 3 vezes para o Callcenter, é viável que seja estabelecido um alerta para cliente que ligam a partir da 4ª, tendo apoio de uma equipe de acolhimento para sanar as dores desses clientes e reduzir a taxa de cancelamento e a reincidência de ligação ao call center.

Clientes com mais de 20 dias de atraso cancelam.
-> Facilitar renegociação.

Todos os clientes com contrato mensal, cancelam.
- Reavaliar a oferta do contrato anual, que fideliza o cliente por mais tempo, fugindo também dos 6 meses de tempo de contrato que tem alta taxa de cancelamento.

Clientes com mais de 15 meses sem interação, tem alta taxa de cancelamento.
-> Automatizar comunicações interativas e estabelecer campanhas periódicas de bônus que forçam a interação do cliente.




Priorização de ofensores.

A Idade, ligação para o callcenter e contrato mensal se mostram ofensores mais prioritários pois conversa diretamente com a entrada de novos clientes, a experiência dos mesmos e abrangem uma parcela grande de nossa base. Portanto, é recomendado que as ações a serem tomadas sejam as ações 1, 4 e 6.




# Panorama após a tomada de ações
Simularemos agora o cenário em que conseguimos converter 30% dos clientes em todos os caos: 50+ não cancelem, não haja mais que 4 ligações no call center e clientes mensais passem para plano anual.


In [ ]:


canceladopriginal = tabela['cancelou'].value_counts(normalize=True).get(1.0, 0.0)

# 1. E se 30% dos clientes com mais de 50 anos não cancelassem?
tabelasimulada = tabela.copy()
filtroidade = (tabelasimulada["idade"] > 50) & (tabelasimulada["cancelou"] == 1.0)
ind_idade = tabelasimulada[filtroidade].sample(frac=0.3, random_state=42).index
tabelasimulada.loc[ind_idade, "cancelou"] = 0.0

canceladosimulado = tabelasimulada['cancelou'].value_counts(normalize=True).get(1.0, 0.0)
print(f"Taxa de Cancelamento Ação 1: {canceladosimulado:.2%}")



# E se tratássemos 30% os clientes que ligam mais de 4 vezes para o Callcenter?
tabelasimulada = tabela.copy()
filtroligacoes = (tabelasimulada["ligacoes_callcenter"] > 4)
ind_ligacoes = tabelasimulada[filtroligacoes].sample(frac=0.3, random_state=42).index
tabelasimulada.loc[ind_ligacoes, "cancelou"] = 0.0
tabelasimulada.loc[ind_ligacoes, "ligacoes_callcenter"] = 4  # Supondo que o tratamento reduza as ligações para 4

canceladosimulado = tabelasimulada['cancelou'].value_counts(normalize=True).get(1.0, 0.0)
print(f"Taxa de Cancelamento Ação 2: {canceladosimulado:.2%}")


# Simular que 30% dos clientes do contrato Mensal passem para o Anual
tabelasimulada = tabela.copy()
filtrocontrato = (tabelasimulada["duracao_contrato"] == "Monthly")
ind_contrato = tabelasimulada[filtrocontrato].sample(frac=0.3, random_state=42).index
tabelasimulada.loc[ind_contrato, "cancelou"] = 0.0
tabelasimulada.loc[ind_contrato, "duracao_contrato"] = "Annual"

canceladosimulado = tabelasimulada['cancelou'].value_counts(normalize=True).get(1.0, 0.0)
print(f"Taxa de Cancelamento Ação 3: {canceladosimulado:.2%}")

# Calcular a redução com as tres ações combinadas
tabelasimulada = tabela.copy()
#Ação 1
ind_idade = tabelasimulada[filtroidade].sample(frac=0.3, random_state=42).index
tabelasimulada.loc[ind_idade, "cancelou"] = 0.0
#Ação 2
ind_ligacoes = tabelasimulada[filtroligacoes].sample(frac=0.3, random_state=42).index
tabelasimulada.loc[ind_ligacoes, "cancelou"] = 0.0
tabelasimulada.loc[ind_ligacoes, "ligacoes_callcenter"] = 4  # Supondo que o tratamento reduza as ligações para 4
#Ação 3
ind_contrato = tabelasimulada[filtrocontrato].sample(frac=0.3, random_state=42).index
tabelasimulada.loc[ind_contrato, "cancelou"] = 0.0
tabelasimulada.loc[ind_contrato, "duracao_contrato"] = "Annual"

canceladosimulado = tabelasimulada['cancelou'].value_counts(normalize=True).get(1.0, 0.0)
print(f"Taxa de Cancelamento Ação Combinada: {canceladosimulado:.2%}")


Taxa de Cancelamento Ação 1: 51.29%
Taxa de Cancelamento Ação 2: 47.13%
Taxa de Cancelamento Ação 3: 50.86%
Taxa de Cancelamento Ação Combinada: 38.04%
